<a href="https://colab.research.google.com/github/pearlnkosi306/FundaAzu/blob/main/stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gikomba Market Stall Performance Analysis

**Author:** [Buhle Pearl Nkosi] · PLP Data Analytics [Cohort PLP]
**Client:** Wanjiku, fresh-produce stall, Gikomba Market
**Tools:** core Python only (no pandas or NumPy)

**Business question:** How is Wanjiku's business doing? Is it growing? What should she restock, and where should she focus?

## Dataset

Two weeks of daily sales (KES) and a stock register. The data cell below is copied exactly as provided in the assignment.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics for the full 14 days

**Approach:** Use `sum()`, `len()`, `max()` and `min()` for the totals. To find *which* day was best or worst, use `list.index()` to get the position, then look up the day name in the parallel `days` list. Positions 0-6 are week 1 and 7-13 are week 2, so the week number is `position // 7 + 1`.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest = max(daily_sales)
lowest = min(daily_sales)

best_pos = daily_sales.index(highest)
worst_pos = daily_sales.index(lowest)
best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales:   KES {total_sales:,}")
print(f"Average daily: KES {average_sales:,.2f}")
print(f"Highest day:   KES {highest:,} ({best_day}, week {best_week})")
print(f"Lowest day:    KES {lowest:,} ({worst_day}, week {worst_week})")

Total sales:   KES 46,450
Average daily: KES 3,317.86
Highest day:   KES 5,600 (Sat, week 2)
Lowest day:    KES 1,950 (Tue, week 1)


## Task 2: Week totals and week-on-week growth

**Approach:** A function slices the list into week 1 (`[:7]`) and week 2 (`[7:14]`), sums each, and returns both as a tuple. The result is unpacked into two variables, then growth is `(week 2 - week 1) / week 1 * 100`.

In [3]:
def week_totals(sales):
    """Return a tuple (week 1 total, week 2 total) from a 14-day sales list."""
    week1 = sum(sales[:7])
    week2 = sum(sales[7:14])
    return week1, week2


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:.1f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.7%


## Task 3: Classify each day as Strong, Normal or Weak

**Approach:** `classify_day()` compares a day's sales with the average. More than 20% above average is "Strong", more than 20% below is "Weak", and everything else is "Normal". A loop applies it to every day, and the dictionary counting pattern `counts[label] = counts.get(label, 0) + 1` tallies each category.

In [4]:
def classify_day(amount, average):
    """Return 'Strong' if amount is >20% above average, 'Weak' if >20% below, else 'Normal'."""
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    else:
        return "Normal"


day_labels = []
counts = {}
for amount in daily_sales:
    label = classify_day(amount, average_sales)
    day_labels.append(label)
    counts[label] = counts.get(label, 0) + 1

for i, (day, amount, label) in enumerate(zip(days, daily_sales, day_labels)):
    print(f"Week {i // 7 + 1} {day}: KES {amount:>5,}  {label}")

print()
print("Day counts:", counts)

Week 1 Mon: KES 2,300  Weak
Week 1 Tue: KES 1,950  Weak
Week 1 Wed: KES 2,800  Normal
Week 1 Thu: KES 2,100  Weak
Week 1 Fri: KES 3,600  Normal
Week 1 Sat: KES 5,200  Strong
Week 1 Sun: KES 4,100  Strong
Week 2 Mon: KES 2,500  Weak
Week 2 Tue: KES 2,200  Weak
Week 2 Wed: KES 3,100  Normal
Week 2 Thu: KES 2,400  Weak
Week 2 Fri: KES 3,900  Normal
Week 2 Sat: KES 5,600  Strong
Week 2 Sun: KES 4,700  Strong

Day counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays versus weekends

**Approach:** Loop through the data and use the day name to decide the group: Sat and Sun are weekend days, the rest are weekdays. Then compare the two averages as a percentage difference relative to the weekday average.

In [5]:
weekday_total, weekday_count = 0, 0
weekend_total, weekend_count = 0, 0

for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_total += amount
        weekend_count += 1
    else:
        weekday_total += amount
        weekday_count += 1

weekday_avg = weekday_total / weekday_count
weekend_avg = weekend_total / weekend_count
weekend_uplift_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekends are {weekend_uplift_pct:.1f}% higher than weekdays")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekends are 82.5% higher than weekdays


## Task 5: Restock list and gross margins

**Approach:** One function loops through the stock dictionary with `.items()` and collects every product whose quantity is at or below its reorder level. A second loop calculates gross margin as `(price - cost) / price * 100` for each product.

In [6]:
def products_to_reorder(stock_register):
    """Return a list of product names whose qty is at or below the reorder level."""
    reorder_list = []
    for name, info in stock_register.items():
        if info["qty"] <= info["reorder"]:
            reorder_list.append(name)
    return reorder_list


def gross_margins(stock_register):
    """Return a dictionary of product name -> gross margin percentage."""
    margins = {}
    for name, info in stock_register.items():
        margins[name] = (info["price"] - info["cost"]) / info["price"] * 100
    return margins


reorder_list = products_to_reorder(stock)
margins = gross_margins(stock)

print("Products to reorder:", reorder_list)
print()
for name, margin in margins.items():
    print(f"{name:<9} gross margin: {margin:.1f}%")

Products to reorder: ['Tomatoes', 'Sukuma']

Tomatoes  gross margin: 33.3%
Onions    gross margin: 30.0%
Sukuma    gross margin: 40.0%
Potatoes  gross margin: 33.3%
Avocado   gross margin: 42.9%


## Task 6: Stall Performance Report

**Approach:** Bring the results from Tasks 1-5 together in one report using f-strings. Money uses `KES {value:,.2f}` for thousands separators, and `:<`, `:>` alignment keeps the columns neat.

In [7]:
line = "=" * 46
print(line)
print("        STALL PERFORMANCE REPORT - GIKOMBA")
print(line)
print(f"{'Total sales (14 days):':<26}KES {total_sales:>12,.2f}")
print(f"{'Average daily sales:':<26}KES {average_sales:>12,.2f}")
print(f"{'Best day:':<26}{best_day} (week {best_week}), KES {highest:,}")
print(f"{'Worst day:':<26}{worst_day} (week {worst_week}), KES {lowest:,}")
print("-" * 46)
print(f"{'Week 1 total:':<26}KES {week1_total:>12,.2f}")
print(f"{'Week 2 total:':<26}KES {week2_total:>12,.2f}")
print(f"{'Week-on-week growth:':<26}{growth_pct:>16.1f}%")
print("-" * 46)
print(f"{'Strong days:':<26}{counts.get('Strong', 0):>16}")
print(f"{'Normal days:':<26}{counts.get('Normal', 0):>16}")
print(f"{'Weak days:':<26}{counts.get('Weak', 0):>16}")
print("-" * 46)
print(f"{'Weekday average:':<26}KES {weekday_avg:>12,.2f}")
print(f"{'Weekend average:':<26}KES {weekend_avg:>12,.2f}")
print(f"{'Weekend uplift:':<26}{weekend_uplift_pct:>16.1f}%")
print("-" * 46)
print("Gross margin by product:")
for name, margin in margins.items():
    print(f"  {name:<24}{margin:>16.1f}%")
print("-" * 46)
print(f"{'Restock now:':<14}{', '.join(reorder_list)}")
print(line)

        STALL PERFORMANCE REPORT - GIKOMBA
Total sales (14 days):    KES    46,450.00
Average daily sales:      KES     3,317.86
Best day:                 Sat (week 2), KES 5,600
Worst day:                Tue (week 1), KES 1,950
----------------------------------------------
Week 1 total:             KES    22,050.00
Week 2 total:             KES    24,400.00
Week-on-week growth:                  10.7%
----------------------------------------------
Strong days:                             4
Normal days:                             4
Weak days:                               6
----------------------------------------------
Weekday average:          KES     2,685.00
Weekend average:          KES     4,900.00
Weekend uplift:                       82.5%
----------------------------------------------
Gross margin by product:
  Tomatoes                            33.3%
  Onions                              30.0%
  Sukuma                              40.0%
  Potatoes                           

## Task 7: Business recommendations

**Approach:** Turn the numbers from Tasks 1-6 into advice Wanjiku can act on. The cell below collects the exact figures behind each recommendation so every claim in the final section can be traced back to the analysis. The recommendations themselves follow in the last section.

In [8]:
weak_days_list = [f"{days[i]} (wk {i // 7 + 1})" for i, label in enumerate(day_labels) if label == "Weak"]

print("Evidence used in the recommendations")
print("-" * 46)
print(f"Restock now:          {', '.join(reorder_list)}")
print(f"Weekend vs weekday:   KES {weekend_avg:,.0f} vs KES {weekday_avg:,.0f} ({weekend_uplift_pct:.1f}% higher)")
print(f"Weak days ({len(weak_days_list)}):         {', '.join(weak_days_list)}")
print(f"Best margins:         Avocado {margins['Avocado']:.1f}%, Sukuma {margins['Sukuma']:.1f}%")
print(f"Growth:               KES {week1_total:,} -> KES {week2_total:,} ({growth_pct:.1f}%)")

Evidence used in the recommendations
----------------------------------------------
Restock now:          Tomatoes, Sukuma
Weekend vs weekday:   KES 4,900 vs KES 2,685 (82.5% higher)
Weak days (6):         Mon (wk 1), Tue (wk 1), Thu (wk 1), Mon (wk 2), Tue (wk 2), Thu (wk 2)
Best margins:         Avocado 42.9%, Sukuma 40.0%
Growth:               KES 22,050 -> KES 24,400 (10.7%)


## Recommendations for Wanjiku

1. **Restock Tomatoes and Sukuma today.** Tomatoes (18 kg) are below their 25 kg reorder level and Sukuma (9 kg) is below its 15 kg level, and both are everyday sellers you cannot afford to run out of on a busy weekend.
2. **Stock up before Friday and focus your effort on weekends.** Saturday and Sunday average KES 4,900 a day against KES 2,685 on weekdays, which is 82.5% higher, and your best day was Saturday of week 2 at KES 5,600.
3. **Fix the quiet weekdays with a promotion.** Six of the 14 days were "Weak" (more than 20% below the KES 3,318 average) and all of them were Mondays, Tuesdays or Thursdays, with the lowest at KES 1,950 on a Tuesday, so a weekday bundle offer such as tomatoes, onions and sukuma together is the biggest opportunity.
4. **Push Avocado and Sukuma, which earn the best margins.** Avocado earns 42.9% and Sukuma 40.0%, compared with 30.0% for Onions, so displaying them first earns more profit per shilling sold.
5. **Your business is growing.** Sales rose from KES 22,050 in week 1 to KES 24,400 in week 2, a 10.7% increase, so keep recording daily sales and re-run this analysis every fortnight to confirm the trend.